Objetivo de la Notebook 2

Esta notebook tendrá como objetivo preparar el conjunto de datos para el entrenamiento de los modelos de análisis de sentimiento mediante la aplicación de distintas técnicas de preprocesamiento de texto. Cada transformación estará justificada a partir de las observaciones realizadas durante el análisis exploratorio.

In [1]:
#Imports

import pandas as pd
import re
import string

import nltk

from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

nltk.download('punkt')
nltk.download('stopwords')

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\31pau\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\31pau\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [2]:
TRAIN_PATH = "../data/raw/training.1600000.processed.noemoticon.csv"

COLUMN_NAMES = [
    "polarity",
    "id",
    "date",
    "query",
    "user",
    "text"
]

df = pd.read_csv(
    TRAIN_PATH,
    encoding="latin-1",
    header=None,
    names=COLUMN_NAMES,
    usecols=["polarity", "text"]
)

In [3]:
#Head

df.head()

,polarity,text
0,0,"@switchfoot http://twitpic.com/2y1zl - Awww, t..."
1,0,is upset that he can't update his Facebook by ...
2,0,@Kenichan I dived many times for the ball. Man...
3,0,my whole body feels itchy and like its on fire
4,0,"@nationwideclass no, it's not behaving at all...."


In [4]:
#Info

df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1600000 entries, 0 to 1599999
Data columns (total 2 columns):
 #   Column    Non-Null Count    Dtype 
---  ------    --------------    ----- 
 0   polarity  1600000 non-null  int64 
 1   text      1600000 non-null  object
dtypes: int64(1), object(1)
memory usage: 24.4+ MB


In [5]:
#Inspección general de los textos

df["text"].sample(10, random_state=42)

541200                @chrishasboobs AHHH I HOPE YOUR OK!!! 
750        @misstoriblack cool , i have no tweet apps  fo...
766711     @TiannaChaos i know  just family drama. its la...
285055     School email won't open  and I have geography ...
705995                                upper airways problem 
379611            Going to miss Pastor's sermon on Faith... 
1189018              on lunch....dj should come eat with me 
667030      @piginthepoke oh why are you feeling like that? 
93541        gahh noo!peyton needs to live!this is horrible 
1097326    @mrstessyman thank you glad you like it! There...
Name: text, dtype: object

In [6]:
#Validación de contenido de URL's

df["text"].str.contains(r"http|www", case=False, regex=True).sum()

np.int64(85971)

In [7]:
#Validacion de contenido Hashtags

df["text"].str.contains(r"#", regex=True).sum()

np.int64(36812)

In [8]:
df[df["text"].str.contains("#")]["text"].sample(10, random_state=42)

14545                                still loading. #asot400
1196640    #Confession I have a crush on a supposed frien...
287645     My brain is fried and I haven't even done anyt...
1264518    @myinnergrownup because #clothdiapers are awes...
742671     lately my weekends are way too short  #exhaust...
1010908    @WilfridDierkes You mean this one ï¿½ ? It's f...
1322535          I get to go on a hovercraft soon  #commutee
490576                 @squarespace Why not me  #squarespace
102606     Struggling with a nasty bug in PHP with '#' in...
674905     #inaperfectworld people would pay me to move m...
Name: text, dtype: object

In [9]:
#Validacion de contenido de numeros

df["text"].str.contains(r"\d", regex=True).sum()

np.int64(366487)

In [10]:
df[df["text"].str.contains(r"\d")]["text"].sample(10, random_state=42)

558732     Off out for a jog in the rain - hoping not to ...
753978     http://lasvegas.craigslist.org/cto/1232876515....
1011415    Off to part 2 of a bereavement class today for...
726991     Journey is not the same this summer as summer ...
562658                   Good morning! 101 heat index today 
1409649    @jenndtl hey did you take the rubric home for ...
445780                I need Jordan and Chocolate  xxx &lt;3
1522448    @joshgroban hi, i saw u at the 'disney music' ...
1150949    Last night descended into pure drunken debauch...
710039     Workingg 6-close  I haaate el chico! Haha if y...
Name: text, dtype: object

In [11]:
#Validacion de caracteres HTML

df["text"].str.contains(r"&\w+;", regex=True).sum()

np.int64(94458)

In [12]:
df[df["text"].str.contains(r"&\w+;")]["text"].sample(10, random_state=42)

149803     Falcon Ridge lineup looking lean this year. I ...
1243799    listenin to @jonasbrothers &quot;fly with me&q...
1402990    Working all day today and then doing some more...
306903     Oh my God. How tragic.  &quot;Pinoy among pass...
121485     @elishacopeland a lil but totally my fault, i ...
960194     enjoying my lazy weekend w/my hubby, maximilli...
509095     sometimes i get scared to tweet, cause lor get...
1313975    hoesef was here! suck my dick zeena$ty  oh &am...
1074799    @Apey 1) It was an obscure &quot;Cruising&quot...
1046736    @its_me_b http://twitpic.com/5hauh - have the ...
Name: text, dtype: object

## Conclusiones del análisis del texto

A partir del análisis exploratorio del contenido de los tweets, se identificaron distintos elementos característicos de este tipo de publicaciones, como menciones a usuarios, URLs, hashtags, entidades HTML, signos de puntuación y palabras escritas con diferentes estilos. En función de estas observaciones, se definió el siguiente pipeline de preprocesamiento:

* **Conversión a minúsculas:** La decisión, luego del análisis, es aplicar la conversión a minúsculas, ya que permite unificar palabras que difieren únicamente por el uso de mayúsculas y reducir el tamaño del vocabulario.

* **Eliminación de URLs:** La decisión es eliminar las URLs, debido a que no aportan información semántica relevante para el análisis de sentimiento y generan una gran cantidad de tokens únicos que incrementan el ruido del conjunto de datos.

* **Eliminación de menciones (`@usuario`):** La decisión es eliminar las menciones, ya que los nombres de usuario no contienen información útil para determinar el sentimiento expresado en el tweet.

* **Tratamiento de hashtags:** La decisión es conservar la palabra asociada al hashtag eliminando únicamente el símbolo `#`, dado que los hashtags suelen contener términos con una importante carga semántica que pueden resultar útiles para la clasificación del sentimiento.

* **Conversión de entidades HTML:** La decisión es convertir las entidades HTML a sus caracteres originales utilizando `html.unescape()`, con el objetivo de preservar el contenido original del texto sin perder información relevante.

* **Eliminación de signos de puntuación:** La decisión es eliminar los signos de puntuación, ya que reducen el ruido del texto y facilitan las etapas posteriores de procesamiento.

* **Eliminación de espacios múltiples:** La decisión es reemplazar múltiples espacios consecutivos por un único espacio para normalizar el formato de los textos y evitar inconsistencias durante el procesamiento.

* **Tokenización:** La decisión es aplicar la tokenización para dividir cada tweet en palabras individuales (tokens), facilitando así las tareas posteriores de procesamiento y representación del texto.

* **Eliminación de stopwords:** La decisión es eliminar las stopwords, debido a que son palabras muy frecuentes que, en términos generales, aportan poca capacidad discriminativa para el análisis de sentimiento.

* **Eliminación de números:** La decisión es no eliminar los números. Durante el análisis se observó que muchos de ellos forman parte del contenido de los tweets y pueden aportar contexto, por lo que se optó por conservarlos.

* **Expansión de contracciones:** La decisión es no expandir las contracciones del idioma inglés (por ejemplo, *don't* o *won't*), con el fin de preservar el contenido original del texto y evitar introducir transformaciones adicionales.

* **Normalización de palabras repetidas:** La decisión es conservar expresiones como *"sooo"* o *"nooo"*, ya que este tipo de construcciones suelen representar énfasis y pueden aportar información relevante sobre el sentimiento expresado por el usuario.

* **Lematización:** La decisión es aplicar lematización con el objetivo de reducir el tamaño del vocabulario y unificar las diferentes formas de una misma palabra en su lema. De esta manera, términos como run, running y ran serán tratados como una única unidad léxica, favoreciendo una representación más consistente del texto para el entrenamiento de los modelos.


## Conversión de texto

Pasos del pipeline de conversión

1)  Conversión a minúsculas.
2)  Eliminación de URLs.
3)  Eliminación de menciones (@usuario).
4)  Eliminación del símbolo # conservando la palabra.
5)  Conversión de entidades HTML mediante html.unescape().
6)  Eliminación de signos de puntuación.
7)  Eliminación de espacios múltiples.
8)  Procesamiento con spaCy (tokenización + lematización).
9)  Eliminación de stopwords (conservando not, no y nor).
10) Reconstrucción del texto.

In [13]:
#Imports

import html
import spacy

In [27]:
# Configuración de spaCy
# Se deshabilitan componentes que no son necesarios para el
# preprocesamiento, mejorando el rendimiento.

nlp = spacy.load(
    "en_core_web_sm",
    disable=["parser", "ner"]
)

# Stopwords en inglés
stop_words = set(stopwords.words("english"))

# Se conservan las palabras de negación por su importancia
# en el análisis de sentimiento.
stop_words = stop_words - {"not", "no", "nor"}

In [15]:
#Funcion elimina URL's

def remove_urls(text):
    return re.sub(r"http\S+|www\S+", "", text)

In [16]:
#Funcion elimina menciones

def remove_mentions(text):
    return re.sub(r"@\w+", "", text)

In [17]:
#Funcion elimina hashtags

def process_hashtags(text):
    return text.replace("#", "")

In [18]:
#Funcion convierte entidades HTML

def decode_html(text):
    return html.unescape(text)

In [19]:
#Funcion elimina signos de puntuación

def remove_punctuation(text):
    return re.sub(
        f"[{re.escape(string.punctuation)}]",
        " ",
        text
    )

In [20]:
#Funcion normalizacion de espacios

def normalize_spaces(text):
    return re.sub(r"\s+", " ", text).strip()

In [21]:
#Conversión

def clean_text(text):

    # Conversión a minúsculas
    text = text.lower()

    # Eliminación de URLs
    text = remove_urls(text)

    # Eliminación de menciones
    text = remove_mentions(text)

    # Conservación de hashtags eliminando solo '#'
    text = process_hashtags(text)

    # Conversión de entidades HTML
    text = decode_html(text)

    # Eliminación de espacios múltiples
    text = normalize_spaces(text)

    return text

In [22]:
#Prueba de la funcion
sample = df["text"].sample(5, random_state=42)

for i, tweet in enumerate(sample, start=1):
    print(f"Tweet {i}")
    print("-" * 80)
    print("Original:")
    print(tweet)
    print()
    print("Limpieza:")
    print(clean_text(tweet))
    print("\n")

Tweet 1
--------------------------------------------------------------------------------
Original:
@chrishasboobs AHHH I HOPE YOUR OK!!! 

Limpieza:
ahhh i hope your ok!!!


Tweet 2
--------------------------------------------------------------------------------
Original:
@misstoriblack cool , i have no tweet apps  for my razr 2

Limpieza:
cool , i have no tweet apps for my razr 2


Tweet 3
--------------------------------------------------------------------------------
Original:
@TiannaChaos i know  just family drama. its lame.hey next time u hang out with kim n u guys like have a sleepover or whatever, ill call u

Limpieza:
i know just family drama. its lame.hey next time u hang out with kim n u guys like have a sleepover or whatever, ill call u


Tweet 4
--------------------------------------------------------------------------------
Original:
School email won't open  and I have geography stuff on there to revise! *Stupid School* :'(

Limpieza:
school email won't open and i have geo

Durante el desarrollo del pipeline se evaluaron distintas estrategias para el tratamiento de la puntuación. Finalmente se decidió delegar esta tarea al procesamiento lingüístico de spaCy, ya que preserva correctamente las contracciones del idioma inglés (por ejemplo, "won't" → "will not"), consideradas de mayor importancia para el análisis de sentimiento que algunos casos aislados de puntuación incorrecta presentes en el conjunto de datos.

In [23]:
#Aplico Pipeline de transformaciones al dataset completo

#df["clean_text"] = df["text"].apply(clean_text)

df["normalized_text"] = df["text"].apply(clean_text)

In [24]:
df[["text", "normalized_text"]].sample(5, random_state=42)

,text,normalized_text
541200,@chrishasboobs AHHH I HOPE YOUR OK!!!,ahhh i hope your ok!!!
750,"@misstoriblack cool , i have no tweet apps fo...","cool , i have no tweet apps for my razr 2"
766711,@TiannaChaos i know just family drama. its la...,i know just family drama. its lame.hey next ti...
285055,School email won't open and I have geography ...,school email won't open and i have geography s...
705995,upper airways problem,upper airways problem


In [25]:
def process_doc(doc):

    tokens = [
        token.lemma_
        for token in doc
        if (
            token.text not in stop_words
            and not token.is_space
            and not token.is_punct
        )
    ]

    return " ".join(tokens)

In [26]:
sample = df["normalized_text"].sample(5, random_state=42)

docs = nlp.pipe(sample)

for i, doc in enumerate(docs, start=1):
    print(f"Tweet {i}")
    print("-" * 80)
    print("Normalizado:")
    print(sample.iloc[i - 1])
    print()
    print("Procesado:")
    print(process_doc(doc))
    print("\n")

Tweet 1
--------------------------------------------------------------------------------
Normalizado:
ahhh i hope your ok!!!

Procesado:
ahhh hope ok


Tweet 2
--------------------------------------------------------------------------------
Normalizado:
cool , i have no tweet apps for my razr 2

Procesado:
cool no tweet app razr 2


Tweet 3
--------------------------------------------------------------------------------
Normalizado:
i know just family drama. its lame.hey next time u hang out with kim n u guys like have a sleepover or whatever, ill call u

Procesado:
know family drama lame.hey next time u hang kim n u guy like sleepover whatever ill call u


Tweet 4
--------------------------------------------------------------------------------
Normalizado:
school email won't open and i have geography stuff on there to revise! *stupid school* :'(

Procesado:
school email will not open geography stuff revise stupid school


Tweet 5
-------------------------------------------------------

In [28]:
processed_texts = []

for doc in nlp.pipe(
    df["normalized_text"],
    batch_size=1000
):
    processed_texts.append(process_doc(doc))

df["clean_text"] = processed_texts

In [29]:
df[
    ["text", "normalized_text", "clean_text"]
].sample(5, random_state=42)

,text,normalized_text,clean_text
541200,@chrishasboobs AHHH I HOPE YOUR OK!!!,ahhh i hope your ok!!!,ahhh hope ok
750,"@misstoriblack cool , i have no tweet apps fo...","cool , i have no tweet apps for my razr 2",cool no tweet app razr 2
766711,@TiannaChaos i know just family drama. its la...,i know just family drama. its lame.hey next ti...,know family drama lame.hey next time u hang ki...
285055,School email won't open and I have geography ...,school email won't open and i have geography s...,school email will not open geography stuff rev...
705995,upper airways problem,upper airways problem,upper airway problem


In [30]:
#Vereficacion de tweets vacios post procesamiento

(df["clean_text"].str.strip() == "").sum()

np.int64(6564)

In [31]:
#Verificacion nulos

df["clean_text"].isnull().sum()

np.int64(0)

In [32]:
#Eliminacion de tweets vacíos

df = df[df["clean_text"].str.strip() != ""].copy()
df.reset_index(drop=True, inplace=True)

In [33]:
#Verificacion del tamaño del dataset

df.shape

(1593436, 4)

In [34]:
#Eliminacion columna intermedia

df.drop(columns=["normalized_text"], inplace=True)

In [35]:
df.head()

,polarity,text,clean_text
0,0,"@switchfoot http://twitpic.com/2y1zl - Awww, t...",be bummer shoulda get david carr third day
1,0,is upset that he can't update his Facebook by ...,upset can not update facebook texte might cry ...
2,0,@Kenichan I dived many times for the ball. Man...,dive many time ball manage save 50 rest go bound
3,0,my whole body feels itchy and like its on fire,whole body feel itchy like fire
4,0,"@nationwideclass no, it's not behaving at all....",no be not behave be mad can not see


In [36]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1593436 entries, 0 to 1593435
Data columns (total 3 columns):
 #   Column      Non-Null Count    Dtype 
---  ------      --------------    ----- 
 0   polarity    1593436 non-null  int64 
 1   text        1593436 non-null  object
 2   clean_text  1593436 non-null  object
dtypes: int64(1), object(2)
memory usage: 36.5+ MB


In [37]:
df = df[["polarity", "clean_text"]]

In [38]:
df.head()


,polarity,clean_text
0,0,be bummer shoulda get david carr third day
1,0,upset can not update facebook texte might cry ...
2,0,dive many time ball manage save 50 rest go bound
3,0,whole body feel itchy like fire
4,0,no be not behave be mad can not see


In [39]:

df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1593436 entries, 0 to 1593435
Data columns (total 2 columns):
 #   Column      Non-Null Count    Dtype 
---  ------      --------------    ----- 
 0   polarity    1593436 non-null  int64 
 1   clean_text  1593436 non-null  object
dtypes: int64(1), object(1)
memory usage: 24.3+ MB


In [40]:
df.to_csv(
    "../data/processed/training_preprocessed.csv",
    index=False
)

In [41]:
import os

os.path.exists("../data/processed/training_preprocessed.csv")

True

## Conclusión del preprocesamiento

Durante esta etapa se desarrolló un pipeline de preprocesamiento orientado al análisis de sentimiento sobre tweets en inglés. El objetivo principal fue transformar los textos originales en una representación más consistente y adecuada para los modelos de clasificación, reduciendo ruido y conservando la información relevante para determinar la polaridad del sentimiento.

Inicialmente se realizó un análisis exploratorio del contenido textual, identificando la presencia de elementos que podían afectar el modelado, como URLs, menciones de usuarios, hashtags, caracteres HTML y números. A partir de este análisis se definieron las transformaciones a aplicar, incluyendo la conversión a minúsculas, eliminación de elementos sin valor semántico, normalización de espacios y tratamiento del contenido textual.

Luego se incorporó un procesamiento lingüístico mediante spaCy, utilizando tokenización y lematización para reducir la variabilidad del vocabulario. En esta etapa se decidió eliminar stopwords, conservando las palabras de negación (`not`, `no` y `nor`) debido a su importancia en tareas de análisis de sentimiento, ya que pueden modificar completamente el significado de una expresión.

Debido al tamaño del conjunto de datos de entrenamiento (1.600.000 registros), se optimizó la ejecución utilizando `nlp.pipe()` de spaCy para procesar los textos en lotes, evitando el procesamiento individual de cada tweet y reduciendo significativamente el tiempo de ejecución sin modificar los resultados obtenidos.

Finalmente, se validó el resultado del pipeline verificando la ausencia de valores nulos y analizando los registros que quedaron sin contenido luego de las transformaciones. Se detectaron 6.564 tweets sin información semántica útil (0,41 % del dataset original), los cuales fueron eliminados antes de la etapa de modelado.

Como resultado final se obtuvo un dataset procesado compuesto por 1.593.436 registros y las variables necesarias para la siguiente etapa del proyecto: la vectorización del texto y el entrenamiento de modelos de clasificación de sentimiento.
